# Excel File Comparison - 7 Sheets
Compares values between **code output** and **sample output** Excel files across all sheets.

In [ ]:
import pandas as pd
import numpy as np

# ── CONFIG ──────────────────────────────────────────────────────────────────
CODE_FILE   = 'code_output.xlsx'    # ← your code output file
SAMPLE_FILE = 'sample_output.xlsx'  # ← your sample/expected output file

# First column is the row label (Category / date / seasoning etc.)
INDEX_COL   = 0      # change if your label column is not the first one
TOLERANCE   = 1e-6   # numeric diff threshold (set to 0 for exact match)
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
code_sheets   = pd.read_excel(CODE_FILE,   sheet_name=None, index_col=INDEX_COL)
sample_sheets = pd.read_excel(SAMPLE_FILE, sheet_name=None, index_col=INDEX_COL)

print('Sheets in code file  :', list(code_sheets.keys()))
print('Sheets in sample file:', list(sample_sheets.keys()))

In [ ]:
def compare_sheets(df_code, df_sample, sheet_name, tol=TOLERANCE):
    """Returns a diff DataFrame and a summary dict for one sheet."""

    # Align on common rows & columns only
    common_rows = df_code.index.intersection(df_sample.index)
    common_cols = df_code.columns.intersection(df_sample.columns)

    a = df_code.loc[common_rows, common_cols]
    b = df_sample.loc[common_rows, common_cols]

    # Build a mask of differing cells
    diff_mask = pd.DataFrame(False, index=common_rows, columns=common_cols)

    for col in common_cols:
        col_a = a[col]
        col_b = b[col]
        numeric = pd.to_numeric(col_a, errors='coerce').notna() | pd.to_numeric(col_b, errors='coerce').notna()

        if numeric.any():
            num_a = pd.to_numeric(col_a, errors='coerce')
            num_b = pd.to_numeric(col_b, errors='coerce')
            diff_mask[col] = (num_a - num_b).abs() > tol
        else:
            diff_mask[col] = col_a.astype(str).str.strip() != col_b.astype(str).str.strip()

    # Handle NaN vs NaN (both NaN = same)
    both_nan = a.isna() & b.isna()
    diff_mask = diff_mask & ~both_nan

    total_cells  = diff_mask.size
    diff_cells   = int(diff_mask.sum().sum())

    # Detailed diff table
    rows, cols_idx = np.where(diff_mask.values)
    records = []
    for r, c in zip(rows, cols_idx):
        records.append({
            'Sheet'      : sheet_name,
            'Row'        : common_rows[r],
            'Column'     : common_cols[c],
            'Code Value' : a.iloc[r, c],
            'Sample Value': b.iloc[r, c],
        })

    diff_df = pd.DataFrame(records)

    summary = {
        'Sheet'              : sheet_name,
        'Common Rows'        : len(common_rows),
        'Common Columns'     : len(common_cols),
        'Total Cells Checked': total_cells,
        'Differing Cells'    : diff_cells,
        'Match %'            : round(100 * (1 - diff_cells / total_cells), 2) if total_cells else 100.0,
        'Extra Rows (code)'  : len(df_code.index.difference(df_sample.index)),
        'Extra Cols (code)'  : len(df_code.columns.difference(df_sample.columns)),
    }
    return diff_df, summary

In [ ]:
all_diffs   = []
all_summary = []

sheets_to_compare = [s for s in code_sheets if s in sample_sheets]
missing_in_sample = [s for s in code_sheets if s not in sample_sheets]
missing_in_code   = [s for s in sample_sheets if s not in code_sheets]

if missing_in_sample:
    print('⚠  Sheets in code but NOT in sample:', missing_in_sample)
if missing_in_code:
    print('⚠  Sheets in sample but NOT in code:', missing_in_code)

for sheet in sheets_to_compare:
    diff_df, summary = compare_sheets(code_sheets[sheet], sample_sheets[sheet], sheet)
    all_diffs.append(diff_df)
    all_summary.append(summary)

summary_df = pd.DataFrame(all_summary)
print('\n=== SUMMARY ACROSS ALL SHEETS ===')
display(summary_df)

In [ ]:
# Overall totals
total_checked = summary_df['Total Cells Checked'].sum()
total_diffs   = summary_df['Differing Cells'].sum()
overall_match = round(100 * (1 - total_diffs / total_checked), 2) if total_checked else 100.0

print(f'Total cells checked : {total_checked}')
print(f'Total differing cells: {total_diffs}')
print(f'Overall match       : {overall_match}%')

In [ ]:
# Show all differing cells in one table
all_diffs_df = pd.concat(all_diffs, ignore_index=True)

if all_diffs_df.empty:
    print('✅ No differences found — files match perfectly!')
else:
    print(f'🔴 {len(all_diffs_df)} differing cell(s) found:\n')
    display(all_diffs_df)

In [ ]:
# (Optional) Filter diffs for a specific sheet
SHEET_FILTER = sheets_to_compare[0]   # ← change to any sheet name

sheet_diffs = all_diffs_df[all_diffs_df['Sheet'] == SHEET_FILTER]
print(f'Diffs in sheet "{SHEET_FILTER}": {len(sheet_diffs)}')
display(sheet_diffs)

In [ ]:
# (Optional) Export diff report to Excel
out_path = 'diff_report.xlsx'

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    if not all_diffs_df.empty:
        all_diffs_df.to_excel(writer, sheet_name='All Diffs', index=False)
        for sheet in sheets_to_compare:
            s_df = all_diffs_df[all_diffs_df['Sheet'] == sheet]
            if not s_df.empty:
                s_df.to_excel(writer, sheet_name=sheet[:31], index=False)

print(f'Diff report saved → {out_path}')